# Week 3: The Forward Diffusion Process & Noise Scheduler
**Diffusion Models from Scratch — Seasons of Code 2026**

### Deliverables Checklist
- [x] `NoiseScheduler` class with linear and cosine schedule support
- [x] Closed-form `q(x_t | x_0)` sampling using the reparameterization trick
- [x] Visualization: one image noised at `t = 0, 100, 250, 500, 750, 999`
- [x] Unit tests verifying `x_T ≈ N(0, I)` (pure Gaussian noise)
- [x] Plot comparing linear vs cosine SNR curves
- [ ] Code pushed to GitHub with updated README

## 0. Setup

In [ ]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. The Math Behind Forward Diffusion

The forward process gradually adds Gaussian noise over `T` timesteps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\, \beta_t \mathbf{I})$$

**Reparameterization trick** — instead of stepping one-by-one, we can jump directly to any timestep `t` in closed form:

$$q(x_t | x_0) = \mathcal{N}(x_t;\, \sqrt{\bar\alpha_t}\, x_0,\, (1-\bar\alpha_t)\mathbf{I})$$

Which means we can sample:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

Where:
- $\beta_t$ = noise schedule (how much noise to add at step t)
- $\alpha_t = 1 - \beta_t$
- $\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$ (cumulative product)

As $t \to T$: $\bar\alpha_T \to 0$, so $x_T \approx \mathcal{N}(0, \mathbf{I})$ — pure noise.

## 2. NoiseScheduler Class

In [ ]:
class NoiseScheduler:
    def __init__(self, T=1000, schedule='linear', beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T = T
        self.schedule = schedule
        self.device = device

        if schedule == 'linear':
            self.betas = torch.linspace(beta_start, beta_end, T, device=device)

        elif schedule == 'cosine':
            steps = T + 1
            t = torch.linspace(0, T, steps, device=device)
            f = torch.cos((t / T + 0.008) / 1.008 * math.pi / 2) ** 2
            alphas_bar = f / f[0]
            betas = 1 - alphas_bar[1:] / alphas_bar[:-1]
            self.betas = betas.clamp(0, 0.999)

        else:
            raise ValueError(f"Unknown schedule '{schedule}'. Choose 'linear' or 'cosine'.")

        self.alphas = 1.0 - self.betas
        self.alphas_bar = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_bar = torch.sqrt(self.alphas_bar)
        self.sqrt_one_minus_alphas_bar = torch.sqrt(1.0 - self.alphas_bar)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = self.sqrt_alphas_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_ab = self.sqrt_one_minus_alphas_bar[t].view(-1, 1, 1, 1)
        return sqrt_ab * x0 + sqrt_one_minus_ab * noise, noise

    def snr(self):
        return self.alphas_bar / (1.0 - self.alphas_bar)


scheduler_linear = NoiseScheduler(T=1000, schedule='linear', device=device)
scheduler_cosine = NoiseScheduler(T=1000, schedule='cosine', device=device)

print('Linear  betas: min={:.6f}  max={:.6f}'.format(
    scheduler_linear.betas.min().item(), scheduler_linear.betas.max().item()))
print('Cosine  betas: min={:.6f}  max={:.6f}'.format(
    scheduler_cosine.betas.min().item(), scheduler_cosine.betas.max().item()))
print('Linear  alpha_bar at T-1: {:.6f}'.format(scheduler_linear.alphas_bar[-1].item()))
print('Cosine  alpha_bar at T-1: {:.6f}'.format(scheduler_cosine.alphas_bar[-1].item()))

## 3. Visualization: One Image Noised at t = 0, 100, 250, 500, 750, 999

In [ ]:
dataset = torchvision.datasets.MNIST('./data', train=True, download=True,
                                      transform=transforms.ToTensor())
x0, _ = dataset[0]
x0 = x0.unsqueeze(0).to(device)

timesteps = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(2, len(timesteps), figsize=(16, 6))

for col, t in enumerate(timesteps):
    t_tensor = torch.tensor([t], device=device)

    xt_linear, _ = scheduler_linear.q_sample(x0, t_tensor)
    xt_cosine, _ = scheduler_cosine.q_sample(x0, t_tensor)

    axes[0, col].imshow(xt_linear[0].squeeze().cpu().clamp(0, 1), cmap='gray')
    axes[0, col].set_title(f't = {t}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(xt_cosine[0].squeeze().cpu().clamp(0, 1), cmap='gray')
    axes[1, col].set_title(f't = {t}', fontsize=10)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Linear', fontsize=11, rotation=90, labelpad=40, va='center')
axes[1, 0].set_ylabel('Cosine', fontsize=11, rotation=90, labelpad=40, va='center')
plt.suptitle('Forward Diffusion: Image at t = 0, 100, 250, 500, 750, 999', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Plot: Linear vs Cosine SNR Curves

**Signal-to-Noise Ratio (SNR)** = $\bar\alpha_t / (1 - \bar\alpha_t)$

- High SNR → image is mostly signal (early timesteps)
- SNR → 0 → pure noise (late timesteps)

The cosine schedule was introduced (Ho et al., improved DDPM) because the linear schedule destroys the image too quickly in early timesteps.

In [ ]:
t_axis = np.arange(1000)
snr_linear = scheduler_linear.snr().cpu().numpy()
snr_cosine = scheduler_cosine.snr().cpu().numpy()
ab_linear  = scheduler_linear.alphas_bar.cpu().numpy()
ab_cosine  = scheduler_cosine.alphas_bar.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(t_axis, ab_linear, label='Linear',  color='royalblue')
axes[0].plot(t_axis, ab_cosine, label='Cosine',  color='tomato')
axes[0].set_title('$\\bar{\\alpha}_t$ (cumulative signal fraction)')
axes[0].set_xlabel('Timestep t')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(t_axis, snr_linear, label='Linear', color='royalblue')
axes[1].plot(t_axis, snr_cosine, label='Cosine', color='tomato')
axes[1].set_title('SNR = $\\bar{\\alpha}_t / (1 - \\bar{\\alpha}_t)$')
axes[1].set_xlabel('Timestep t')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(t_axis, np.log10(snr_linear + 1e-10), label='Linear', color='royalblue')
axes[2].plot(t_axis, np.log10(snr_cosine + 1e-10), label='Cosine', color='tomato')
axes[2].set_title('log10(SNR) — easier to read')
axes[2].set_xlabel('Timestep t')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5, label='SNR = 1')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Linear vs Cosine Schedule — SNR Comparison', fontsize=13)
plt.tight_layout()
plt.show()

print('Linear: alpha_bar drops below 0.01 at t =', (ab_linear < 0.01).argmax())
print('Cosine: alpha_bar drops below 0.01 at t =', (ab_cosine < 0.01).argmax())

## 5. Unit Tests: Verify x_T ≈ N(0, I)

At `t = T-1 = 999`, `alpha_bar_T ≈ 0`, so:
$$x_T = \sqrt{\bar\alpha_T}\,x_0 + \sqrt{1-\bar\alpha_T}\,\epsilon \approx \epsilon \sim \mathcal{N}(0, I)$$

We verify this statistically: the final noised images should have mean ≈ 0 and std ≈ 1.

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=512, shuffle=True)
batch_x0, _ = next(iter(loader))
batch_x0 = batch_x0.to(device)

t_final = torch.full((batch_x0.size(0),), 999, dtype=torch.long, device=device)

def run_unit_tests(scheduler, name):
    print(f'\n=== Unit Tests: {name} schedule ===')

    xT, _ = scheduler.q_sample(batch_x0, t_final)
    mean = xT.mean().item()
    std  = xT.std().item()

    print(f'  x_T mean: {mean:.4f}  (expected ≈ 0.0)')
    print(f'  x_T std:  {std:.4f}  (expected ≈ 1.0)')

    test1 = abs(mean) < 0.05
    test2 = abs(std - 1.0) < 0.05
    print(f'  [PASS] mean ≈ 0' if test1 else f'  [FAIL] mean = {mean:.4f}')
    print(f'  [PASS] std  ≈ 1' if test2 else f'  [FAIL] std  = {std:.4f}')

    t0 = torch.zeros(batch_x0.size(0), dtype=torch.long, device=device)
    x0_check, _ = scheduler.q_sample(batch_x0, t0)
    ab0 = scheduler.alphas_bar[0].item()
    expected_mean_scale = math.sqrt(ab0)
    actual_mean_scale   = x0_check.mean().item() / batch_x0.mean().item()
    test3 = abs(actual_mean_scale - expected_mean_scale) < 0.05
    print(f'  [PASS] t=0 barely adds noise (alpha_bar[0]={ab0:.4f})' if test3
          else f'  [FAIL] t=0 test failed')

    corr = torch.corrcoef(torch.stack([xT.flatten(), batch_x0.flatten()]))[0, 1].item()
    test4 = abs(corr) < 0.05
    print(f'  [PASS] x_T uncorrelated with x_0 (corr={corr:.4f})' if test4
          else f'  [FAIL] x_T still correlated with x_0 (corr={corr:.4f})')

    return xT

xT_linear = run_unit_tests(scheduler_linear, 'Linear')
xT_cosine = run_unit_tests(scheduler_cosine, 'Cosine')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

xT_flat = xT_linear.cpu().flatten().numpy()
ref_normal = np.random.randn(len(xT_flat))

axes[0].hist(xT_flat, bins=100, density=True, alpha=0.7, label='x_T (linear)', color='royalblue')
axes[0].hist(ref_normal, bins=100, density=True, alpha=0.5, label='N(0,1)', color='gray')
axes[0].set_title('Distribution of x_T (linear) vs N(0,1)')
axes[0].legend()
axes[0].grid(alpha=0.3)

xT_cos_flat = xT_cosine.cpu().flatten().numpy()
axes[1].hist(xT_cos_flat, bins=100, density=True, alpha=0.7, label='x_T (cosine)', color='tomato')
axes[1].hist(ref_normal, bins=100, density=True, alpha=0.5, label='N(0,1)', color='gray')
axes[1].set_title('Distribution of x_T (cosine) vs N(0,1)')
axes[1].legend()
axes[1].grid(alpha=0.3)

sorted_data = np.sort(xT_flat)
sorted_ref  = np.sort(ref_normal)
min_len = min(len(sorted_data), len(sorted_ref))
axes[2].scatter(sorted_ref[:min_len:100], sorted_data[:min_len:100], s=1, alpha=0.5, color='royalblue')
lims = [min(sorted_ref.min(), sorted_data.min()), max(sorted_ref.max(), sorted_data.max())]
axes[2].plot(lims, lims, 'r--', linewidth=1)
axes[2].set_title('Q-Q Plot: x_T vs N(0,1)')
axes[2].set_xlabel('Theoretical N(0,1)')
axes[2].set_ylabel('x_T quantiles')
axes[2].grid(alpha=0.3)

plt.suptitle('Unit Test Visualization: x_T ≈ N(0, I)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Why Cosine > Linear?

The linear schedule adds noise too aggressively in early timesteps — the image becomes unrecognizable at `t ≈ 500` while still having 500 steps left that are "wasted" on already-destroyed images.

The cosine schedule (Nichol & Dhariwal, 2021) decays $\bar\alpha_t$ more gradually in early steps and more aggressively near the end — giving the model more meaningful training signal throughout.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
sample_img, _ = dataset[7]
sample_img = sample_img.unsqueeze(0).to(device)

ts = [0, 100, 200, 400, 600, 999]
for col, t in enumerate(ts):
    t_t = torch.tensor([t], device=device)
    xl, _ = scheduler_linear.q_sample(sample_img, t_t)
    xc, _ = scheduler_cosine.q_sample(sample_img, t_t)

    snr_l = scheduler_linear.snr()[t].item()
    snr_c = scheduler_cosine.snr()[t].item()

    axes[0, col].imshow(xl[0].squeeze().cpu().clamp(0, 1), cmap='gray')
    axes[0, col].set_title(f't={t}\nSNR={snr_l:.3f}', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(xc[0].squeeze().cpu().clamp(0, 1), cmap='gray')
    axes[1, col].set_title(f't={t}\nSNR={snr_c:.3f}', fontsize=8)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Linear', fontsize=11, rotation=90, labelpad=40, va='center')
axes[1, 0].set_ylabel('Cosine', fontsize=11, rotation=90, labelpad=40, va='center')
plt.suptitle('Linear vs Cosine: Image Degradation + SNR at each timestep', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Summary

| Concept | Formula |
|---|---|
| Forward step | $q(x_t|x_{t-1}) = \mathcal{N}(\sqrt{1-\beta_t}\,x_{t-1},\,\beta_t I)$ |
| Closed-form (reparameterization) | $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$ |
| Terminal distribution | $x_T \approx \mathcal{N}(0, I)$ |
| SNR | $\bar\alpha_t / (1 - \bar\alpha_t)$ |

The reparameterization trick is what makes DDPM training efficient — in Week 4, the UNet is trained to predict $\epsilon$ directly from $x_t$ and $t$, and we never need to run the full Markov chain step by step.